In [ ]:
# Resolve defaults from this project's root, independently of the working directory.
from pathlib import Path as _KSPath
import sys as _ks_sys
from functools import partial as _ks_partial
_ks_start = _KSPath.cwd()
_ks_root = next((p for p in (_ks_start, *_ks_start.parents) if (p / 'kinspec_paths.py').is_file()), None)
if _ks_root is None:
    raise RuntimeError('Cannot locate the KinSPEC project root containing kinspec_paths.py. Start the notebook kernel from the project directory or a subdirectory.')
_ks_sys.path.insert(0, str(_ks_root))
from kinspec_paths import ROOT as KINSPEC_ROOT, project_path as _ks_project_path
_kp = _ks_partial(_ks_project_path, script_dir='others/baseline/HitScreen/script')
_ks_sys.path.insert(0, str(KINSPEC_ROOT / 'external/HitScreen'))
import os as _ks_os
_ks_os.chdir(KINSPEC_ROOT / 'others/baseline/HitScreen/script')



In [ ]:
"""
HitScreen evaluation across multiple models.
- Run each of the five train_valid_1 through train_valid_5 best_model checkpoints on the same test set.
- Group by Target and calculate ROC-AUC, PR-AUC, BEDROC, and EF at 2%, 5%, and 10%.
- Aggregate means and standard deviations.
"""

import os
import re
import glob
import ctypes
import argparse
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torch.nn import Sigmoid
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import rankdata

# Preload CUDA libraries using the existing logic
conda_prefix = os.environ.get('CONDA_PREFIX', '')
if conda_prefix:
    nvidia_lib_dir = f"{conda_prefix}/lib/python3.9/site-packages/nvidia"
    load_order = ['libcudart', 'libcublas', 'libcublasLt', 'libcusparse',
                  'libcufft', 'libcurand', 'libcusolver', 'libnvrtc']
    so_map = {}
    for root, dirs, files in os.walk(nvidia_lib_dir):
        if root.endswith('/lib'):
            for fname in os.listdir(root):
                if fname.endswith('.so') or '.so.' in fname:
                    for prefix in load_order:
                        if fname.startswith(prefix):
                            full_path = os.path.join(root, fname)
                            if prefix not in so_map or len(fname) < len(os.path.basename(so_map[prefix])):
                                so_map[prefix] = full_path
    for prefix in load_order:
        if prefix in so_map:
            try:
                ctypes.CDLL(so_map[prefix], mode=ctypes.RTLD_GLOBAL)
                print(f"✅ Loaded: {os.path.basename(so_map[prefix])}")
            except OSError as e:
                print(f"⚠️  Skipping {prefix}: {e}")

import dgl
print("✅ DGL loaded successfully")

from models import HitScreen
from utils import graph_collate_func
from dataloader_sql import DTIDataset


# ══════════════════════════════════════════════════════════════
# Configuration
# ══════════════════════════════════════════════════════════════
band_range = '0.10_0.50'
band_range_2 = '10_50'

BAND_DIR      = _kp(f'model/HitScreen/data_split_from_merged_true_fold/band_0.10_0.50/train_valid_split')          # Parent directory containing train_valid_1 through train_valid_5
# INPUT_CSV     = f"/home/s1856122/dataset/KIBA/SPECTRA_comparison_from_merged/KIBA_10_50/KIBA_SPECTRA_VSDS_10_50_from_merged_remove_overlap_data_t0.80_cleavage_spectra_unseen_target.csv"
INPUT_CSV     = _kp(f'dataset/KIBA_seen_LUDe_augment_sim08.csv')
OUTPUT_CSV    = _kp(f'others/baseline/HitScreen/results/KIBA_seen_LUDe_augment/evaluation_metrics_KIBA_from_merged_seen_target_augment.csv')

SMILES_DB     = _kp('dataset/preprocessed/HitScreen/KIBA_seen_LUDe_augment/Uni-Mol_molecule_embeddings_seen_augment_ligands.db')
PROTEIN_DB    = _kp('dataset/preprocessed/HitScreen/KIBA_seen_LUDe_augment/Ankh_Large_target_protein_embeddings_seen_target_augment.db')

DEVICE_STR    = "cuda:0"
LABEL_COL     = "Y"     # CSV label column containing 0/1; update if needed
KIBA_COL      = "KIBA"  
KIBA_THRESHOLD = 3.0  
TARGET_COL    = "uniprot_id" # CSV target column; update if needed
BEDROC_ALPHA  = 80.5        # Common BEDROC alpha value


# ══════════════════════════════════════════════════════════════
# Utility functions
# ══════════════════════════════════════════════════════════════

def build_args_from_config(config_path: str) -> argparse.Namespace:
    args = {}
    with open(config_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or ':' not in line:
                continue
            key, _, value = line.partition(':')
            key, value = key.strip(), value.strip()
            if value.lower() == 'true':
                args[key] = True
            elif value.lower() == 'false':
                args[key] = False
            elif value.startswith('[') and value.endswith(']'):
                inner = value[1:-1].split(',')
                try:
                    args[key] = [int(v.strip()) for v in inner]
                except ValueError:
                    args[key] = [v.strip() for v in inner]
            else:
                try:
                    args[key] = int(value)
                except ValueError:
                    try:
                        args[key] = float(value)
                    except ValueError:
                        args[key] = value
    return argparse.Namespace(**args)


def find_best_model(result_dir: str) -> str:
    """Find best_model_epoch_*.pth under result/ and return its path."""
    pattern = os.path.join(result_dir, "best_model_epoch_*.pth")
    files = glob.glob(pattern)
    if not files:
        raise FileNotFoundError(f"Model file not found: {pattern}")
    # If multiple checkpoints exist, choose the highest epoch
    files.sort(key=lambda p: int(re.search(r'epoch_(\d+)', p).group(1)))
    return files[-1]


def compute_bedroc(y_true: np.ndarray, y_score: np.ndarray, alpha: float = 80.5) -> float:
    y_true  = np.asarray(y_true)
    y_score = np.asarray(y_score)
    N = len(y_true)
    n_act = np.sum(y_true == 1)
    if n_act == 0:
        return 0.0

    order  = np.argsort(y_score)[::-1]
    ranks  = np.arange(1, N + 1)[y_true[order] == 1]   # 1-based
    x      = (ranks - 0.5) / N                          # Rank midpoint

    C      = (1.0 - np.exp(-alpha)) / alpha
    rie    = np.mean(np.exp(-alpha * x)) / C

    x_max  = (np.arange(1, n_act + 1) - 0.5) / N
    x_min  = (np.arange(N - n_act + 1, N + 1) - 0.5) / N
    rie_max = np.mean(np.exp(-alpha * x_max)) / C
    rie_min = np.mean(np.exp(-alpha * x_min)) / C

    return float((rie - rie_min) / (rie_max - rie_min))


def compute_ef(y_true: np.ndarray, y_score: np.ndarray, fraction: float) -> float:
    """Pass fraction as a decimal, such as 0.02 for 2%."""
    y_true  = np.asarray(y_true)
    y_score = np.asarray(y_score)
    N = len(y_true)
    n_active = np.sum(y_true == 1)
    if n_active == 0:
        return float('nan')

    percentage  = fraction * 100
    n_selected  = max(1, int(N * percentage / 100))      # Use int rather than ceil, matching the reference code

    sorted_indices = np.argsort(y_score)[::-1]
    sorted_labels  = y_true[sorted_indices]
    n_active_in_selected = np.sum(sorted_labels[:n_selected])

    ef_value = (n_active_in_selected / n_selected) / (n_active / N)
    return float(ef_value)



def evaluate_per_target(df_group: pd.DataFrame, prob_col: str) -> dict:
    """Calculate all metrics for one target."""
    y_true  = df_group[LABEL_COL].values.astype(int)
    y_score = df_group[prob_col].values

    n        = len(y_true)
    n_active = y_true.sum()
    ratio    = n_active / n if n > 0 else 0.0

    # Handle single-class input
    if n_active == 0 or n_active == n:
        return dict(N=n, Actives=n_active, ActiveRatio=ratio,
                    ROC_AUC=np.nan, PR_AUC=np.nan, BEDROC=np.nan,
                    EF_2=np.nan, EF_5=np.nan, EF_10=np.nan)

    roc_auc = roc_auc_score(y_true, y_score)
    pr_auc  = average_precision_score(y_true, y_score)
    bedroc  = compute_bedroc(y_true, y_score, BEDROC_ALPHA)
    ef05    = compute_ef(y_true, y_score, 0.005)
    ef1     = compute_ef(y_true, y_score, 0.01)
    ef2     = compute_ef(y_true, y_score, 0.02)
    ef5     = compute_ef(y_true, y_score, 0.05)
    ef10    = compute_ef(y_true, y_score, 0.10)

    return dict(N=n, Actives=n_active, ActiveRatio=ratio,
                ROC_AUC=roc_auc, PR_AUC=pr_auc, BEDROC=bedroc,
                EF_05=ef05, EF_1=ef1,EF_2=ef2, EF_5=ef5, EF_10=ef10)


# ══════════════════════════════════════════════════════════════
# Main workflow
# ══════════════════════════════════════════════════════════════

device = torch.device(DEVICE_STR if torch.cuda.is_available() and 'cuda' in DEVICE_STR else 'cpu')
print(f"Device: {device}")

# Read the test set once
df_test = pd.read_csv(INPUT_CSV)
# Generate labels from the KIBA column
df_test[LABEL_COL] = (df_test[KIBA_COL] <= KIBA_THRESHOLD).astype(int)
print(f"📂 Test set: {INPUT_CSV}, {len(df_test)} samples")
print(f"   Active (KIBA <= {KIBA_THRESHOLD}): {df_test[LABEL_COL].sum()}")
print(f"   Inactive (KIBA > {KIBA_THRESHOLD}): {(df_test[LABEL_COL] == 0).sum()}")

#===============================check atom number============================
from rdkit import Chem

max_atoms = 0
problem_rows = []
smiles_col = 'SMILES'  # Update this if the column name differs

for idx, smi in enumerate(df_test[smiles_col]):
    try:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is not None:
            n = mol.GetNumAtoms()
            if n > max_atoms:
                max_atoms = n
            if n > 300:
                problem_rows.append((idx, smi, n))
    except:
        pass

print(f"Maximum atom count in the test set: {max_atoms}")
print(f"Molecules with more than 300 atoms: {len(problem_rows)}")
for idx, smi, n in problem_rows[:10]:
    print(f"  Row {idx}: {n} atoms | {str(smi)[:80]}")

# =============remove more than 300 atoms=========

before = len(df_test)
df_test = df_test[df_test['SMILES'].apply(
    lambda smi: Chem.MolFromSmiles(str(smi)) is not None and 
                Chem.MolFromSmiles(str(smi)).GetNumAtoms() <= 300
)].reset_index(drop=True)
print(f"before filter: {before}, after filter: {len(df_test)}, remove: {before - len(df_test)} rows")

# ======================================================


# Locate the five train_valid directories
fold_dirs = sorted(glob.glob(os.path.join(BAND_DIR, "train_valid_*")))
print(f"Found {len(fold_dirs)} folds: {[os.path.basename(d) for d in fold_dirs]}")

sigmoid = Sigmoid()

# Collect per-target metrics for each fold: {target -> [metric_dict, ...]}
from collections import defaultdict
target_metrics = defaultdict(list)  # target -> list of metric dicts (one per fold)

for fold_dir in fold_dirs:
    result_dir  = os.path.join(fold_dir, "result")
    config_path = os.path.join(result_dir, "config.txt")
    model_path  = find_best_model(result_dir)

    print(f"\n{'='*60}")
    print(f"▶ Fold: {os.path.basename(fold_dir)}")
    print(f"  Model: {model_path}")

    args = build_args_from_config(config_path)

    # Construct the dataset and loader
    dataset = DTIDataset(
        list_IDs=df_test.index.values,
        df=df_test,
        protein_language_model_embedding_dim=args.protein_language_model_embedding_dim,
        smiles_db_path=SMILES_DB,
        protein_db_path=PROTEIN_DB,
        is_LigandGCN=args.is_LigandGCN,
        is_unimol_Ligand=args.is_unimol_Ligand,
        is_ProteinCNN=args.is_ProteinCNN,
        is_esm_Protein=args.is_esm_Protein,
        max_drug_nodes=args.ligand_max_nodes,
        protein_padding_length=args.protein_padding_length,
        drug_embedding_dim=args.drug_embedding_dim,
    )
    loader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=getattr(args, 'num_workers', 0),
        drop_last=False,
        collate_fn=graph_collate_func,
    )

    # Load the model
    model = HitScreen(args).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # Inference
    all_probs = []
    with torch.no_grad():
        for v_d, v_d_atomic_embedding, v_p_cnn, v_p_esm, labels in loader:
            v_d                  = v_d.to(device)                  if v_d is not None else None
            v_d_atomic_embedding = v_d_atomic_embedding.to(device) if v_d_atomic_embedding is not None else None
            v_p_cnn              = v_p_cnn.to(device)              if v_p_cnn is not None else None
            v_p_esm              = v_p_esm.to(device)              if v_p_esm is not None else None

            _, score = model(v_d, v_d_atomic_embedding, v_p_cnn, v_p_esm, mode="train")
            probs = sigmoid(score).squeeze(-1).cpu().numpy()
            all_probs.extend(probs.tolist())

    # Release GPU memory
    del model
    torch.cuda.empty_cache()

    # Add prediction probabilities and calculate per-target metrics
    prob_col = f"prob_{os.path.basename(fold_dir)}"
    df_test[prob_col] = all_probs

    for target, grp in df_test.groupby(TARGET_COL):
        metrics = evaluate_per_target(grp, prob_col)
        target_metrics[target].append(metrics)

    print(f"  ✅ Inference complete; probabilities written to column: {prob_col}")


# ══════════════════════════════════════════════════════════════
# Aggregate means and standard deviations across five folds
# ══════════════════════════════════════════════════════════════

rows = []
for target in sorted(target_metrics.keys()):
    fold_results = target_metrics[target]

    # N, Actives, and ActiveRatio are identical across folds; use the first fold's values
    N           = fold_results[0]['N']
    n_actives   = fold_results[0]['Actives']
    active_ratio = fold_results[0]['ActiveRatio']
    baseline_pr = active_ratio   # random baseline for PR-AUC

    def _mean_std(key):
        vals = [d[key] for d in fold_results if not np.isnan(d[key])]
        if not vals:
            return np.nan, np.nan
        return float(np.mean(vals)), float(np.std(vals, ddof=0))

    roc_mean, roc_std   = _mean_std('ROC_AUC')
    pr_mean,  pr_std    = _mean_std('PR_AUC')
    bdr_mean, bdr_std   = _mean_std('BEDROC')
    ef05_mean, ef05_std = _mean_std('EF_05')
    ef1_mean,  ef1_std  = _mean_std('EF_1')
    ef2_mean, ef2_std   = _mean_std('EF_2')
    ef5_mean, ef5_std   = _mean_std('EF_5')
    ef10_mean, ef10_std = _mean_std('EF_10')

    pr_minus_baseline   = pr_mean - baseline_pr if not np.isnan(pr_mean) else np.nan
    # normalized PR-AUC: (PR_AUC - baseline) / (1 - baseline)
    norm_pr = (pr_mean - baseline_pr) / (1 - baseline_pr) if (not np.isnan(pr_mean) and baseline_pr < 1) else np.nan

    rows.append({
        'Target'               : target,
        'N'                    : N,
        'Actives'              : n_actives,
        'ActiveRatio'          : round(active_ratio, 4),
        'ROC_AUC_mean'         : round(roc_mean,  4),
        'ROC_AUC_std'          : round(roc_std,   4),
        'PR_AUC_mean'          : round(pr_mean,   4),
        'PR_AUC_std'           : round(pr_std,    4),
        'PR_AUC_mean_minus_baseline': round(pr_minus_baseline, 4),
        'normalized_PR_AUC'    : round(norm_pr,   4),
        'BEDROC_mean'          : round(bdr_mean,  4),
        'BEDROC_std'           : round(bdr_std,   4),
        'EF_0.5%_mean'         : round(ef05_mean, 4),
        'EF_0.5%_std'          : round(ef05_std,  4),
        'EF_1%_mean'           : round(ef1_mean,  4),
        'EF_1%_std'            : round(ef1_std,   4),
        'EF_2%_mean'           : round(ef2_mean,  4),
        'EF_2%_std'            : round(ef2_std,   4),
        'EF_5%_mean'           : round(ef5_mean,  4),
        'EF_5%_std'            : round(ef5_std,   4),
        'EF_10%_mean'          : round(ef10_mean, 4),
        'EF_10%_std'           : round(ef10_std,  4),
    })



df_out = pd.DataFrame(rows)
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

# ══════════════════════════════════════════════════════════════
# Evaluate each fold on the complete test set, then calculate means and standard deviations
# ══════════════════════════════════════════════════════════════

overall_metrics = []

for fold_dir in fold_dirs:
    prob_col = f"prob_{os.path.basename(fold_dir)}"
    y_true   = df_test[LABEL_COL].values.astype(int)
    y_score  = df_test[prob_col].values

    roc_auc  = roc_auc_score(y_true, y_score)
    pr_auc   = average_precision_score(y_true, y_score)
    baseline = y_true.mean()
    bedroc   = compute_bedroc(y_true, y_score, BEDROC_ALPHA)
    ef05     = compute_ef(y_true, y_score, 0.005)
    ef1      = compute_ef(y_true, y_score, 0.01)
    ef2      = compute_ef(y_true, y_score, 0.02)
    ef5      = compute_ef(y_true, y_score, 0.05)
    ef10     = compute_ef(y_true, y_score, 0.10)

    overall_metrics.append({
        'fold'                      : os.path.basename(fold_dir),
        'ROC_AUC'                   : roc_auc,
        'PR_AUC'                    : pr_auc,
        'PR_AUC_minus_baseline'     : pr_auc - baseline,
        'normalized_PR_AUC'         : (pr_auc - baseline) / (1 - baseline),
        'BEDROC'                    : bedroc,
        'EF_0.5%'                   : ef05,
        'EF_1%'                     : ef1,
        'EF_2%'                     : ef2,
        'EF_5%'                     : ef5,
        'EF_10%'                    : ef10,
    })

df_overall = pd.DataFrame(overall_metrics)

print("\n" + "="*60)
print("📊 Overall evaluation (entire test set, 5 folds)")
print("="*60)
print(df_overall.to_string(index=False))

print("\n📊 Overall evaluation mean ± std")
print("="*60)
metric_cols_overall = [
    'ROC_AUC', 'PR_AUC', 'PR_AUC_minus_baseline', 'normalized_PR_AUC',
    'BEDROC', 'EF_0.5%', 'EF_1%', 'EF_2%', 'EF_5%', 'EF_10%'
]
for col in metric_cols_overall:
    vals = df_overall[col].values
    print(f"  {col:<30s}: {np.mean(vals):.4f} ± {np.std(vals, ddof=0):.4f}")
print("="*60)

# Also save overall evaluation metrics to CSV
overall_summary_path = OUTPUT_CSV.replace('.csv', '_overall.csv')
df_overall.to_csv(overall_summary_path, index=False)
print(f"✅ Overall evaluation saved to: {overall_summary_path}")


df_out.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ Evaluation complete; results saved to: {OUTPUT_CSV}")
print(df_out.to_string(index=False))

✅ DGL loaded successfully
Device: cuda:0
📂 Test set: /home/s1856122/dataset/KIBA/SPECTRA_comparison_from_merged/KIBA_10_50/KIBA_10_50_from_merged_augment/KIBA_SPECTRA_VSDS_10_50_from_merged_remove_overlap_data_t0.80_cleavage_spectra_unseen_target_mol_deepcoy_augment.csv, 20956 samples
   Active (KIBA <= 3.0): 856
   Inactive (KIBA > 3.0): 20100
Maximum atom count in the test set: 96
Molecules with more than 300 atoms: 0
before filter: 20956, after filter: 20956, remove: 0 rows
Found 5 folds: ['train_valid_1', 'train_valid_2', 'train_valid_3', 'train_valid_4', 'train_valid_5']

▶ Fold: train_valid_1
  Model: ./SPECTRA_from_merged/band_0.10_0.50/train_valid_1/result/best_model_epoch_18.pth
  ✅ Inference complete; probabilities written to column: prob_train_valid_1

▶ Fold: train_valid_2
  Model: ./SPECTRA_from_merged/band_0.10_0.50/train_valid_2/result/best_model_epoch_20.pth
  ✅ Inference complete; probabilities written to column: prob_train_valid_2

▶ Fold: train_valid_3
  Model: ./SPE